# Harness Evaluation 可视化分析

In [11]:
from collections import Counter
from pathlib import Path
from statistics import mean
import html
import json
import sys

from IPython.display import HTML, display


def locate_analysis_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / 'reports' / 'analysis',
        Path('/Users/luowei/project/ai-architecture-integrity-study/reports/analysis'),
    ]
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'harness_analysis.py').exists():
            return candidate
    raise FileNotFoundError('Cannot locate reports/analysis/harness_analysis.py')


ANALYSIS_DIR = locate_analysis_dir()
REPORTS_DIR = ANALYSIS_DIR.parent
EXPERIMENTS_DIR = REPORTS_DIR / 'experiments'
sys.path.insert(0, str(ANALYSIS_DIR))

import harness_analysis as ha

evaluation_paths = ha.discover_evaluations(EXPERIMENTS_DIR, include_reruns=False)
runs, constraints, metrics = ha.load_data(evaluation_paths, EXPERIMENTS_DIR)

print(f'Analysis directory: {ANALYSIS_DIR}')
print(f'Loaded {len(runs)} evaluations from {len(evaluation_paths)} JSON files.')

Analysis directory: /Users/luowei/project/ai-architecture-integrity-study/reports/analysis
Loaded 12 evaluations from 12 JSON files.


## 1. harness 获取数据质量

In [12]:
summary = {
    'sessions': len({row['session_id'] for row in runs}),
    'evaluations': len(runs),
    'tasks': sorted({row['task_id'] for row in runs}, key=ha.task_sort_key),
    'constraint_findings': len(constraints),
    'metric_observations': len(metrics),
    'metric_errors': sum(row['status'] == 'error' for row in metrics),
    'average_metric_coverage': mean(row['metric_coverage'] for row in runs),
    'overall_statuses': dict(Counter(row['status'] for row in runs)),
}
display(HTML(
    '<h3>Dataset summary</h3>'
    f"<ul><li>Sessions: {summary['sessions']}</li>"
    f"<li>Evaluations: {summary['evaluations']}</li>"
    f"<li>Tasks: {', '.join(summary['tasks'])}</li>"
    f"<li>Constraint findings: {summary['constraint_findings']}</li>"
    f"<li>Metric observations/errors: {summary['metric_observations']} / {summary['metric_errors']}</li>"
    f"<li>Average metric coverage: {summary['average_metric_coverage']:.1%}</li>"
    f"<li>Overall statuses: {html.escape(str(summary['overall_statuses']))}</li></ul>"
))
display(HTML(ha.make_run_table(runs)))

Session,Task,Condition,Overall,Backend,Frontend,Cross,BE findings,FE findings,Cross findings,Metric coverage,Metric errors,Runtime (s)
08-12 22:08,T1,claude / minimal,partial,failed,error,none,1,0,0,100%,0,2.45
08-12 22:08,T2,claude / minimal,partial,failed,error,none,7,0,0,100%,0,3.16
08-12 22:08,T3,claude / minimal,partial,error,failed,failed,13,36,2,95%,1,5.92
08-13 15:03,T1,codex / minimal,partial,failed,error,none,2,0,0,100%,0,3.37
08-13 15:03,T2,codex / minimal,partial,failed,error,none,10,0,0,100%,0,2.36
08-13 15:03,T3,codex / minimal,partial,error,failed,failed,22,36,2,95%,1,5.42
08-13 20:34,T1,claude / structured,partial,completed,error,none,0,0,0,100%,0,2.70
08-13 20:34,T2,claude / structured,partial,error,error,none,5,0,0,89%,1,3.56
08-13 20:34,T3,claude / structured,partial,error,failed,failed,5,33,7,95%,1,5.47
08-13 21:16,T1,codex / structured,partial,error,error,none,2,0,0,89%,1,3.97


### 讨论

当前数据包含 4 个 session、12 次 evaluation 和 241 条 constraint finding。平均 metric coverage 为 95.6%，但 12 次 evaluation 的总体状态全部为 `partial`，131 个 metric observation 中有 7 个执行错误。

TODO: 查找指标错误原因 + 纳入 baseline 的数据进行评估

## 2. Session 内的任务演化

T1 → T2 → T3 是否持续上升

In [13]:
sessions = sorted({row['session_id'] for row in runs})
tasks = sorted({row['task_id'] for row in runs}, key=ha.task_sort_key)
run_by_key = {(row['session_id'], row['task_id']): row for row in runs}

trajectory_series = []
for session in sessions:
    condition = next(row for row in runs if row['session_id'] == session)
    label = f"{condition['agent']}/{condition['strategy']}"
    values = [run_by_key.get((session, task), {}).get('total_findings') for task in tasks]
    trajectory_series.append((label, values))

display(HTML(ha.svg_line_chart(
    '各实验条件的约束问题演化',
    tasks,
    trajectory_series,
    'Constraint findings (count)',
)))

### 讨论

四个条件都呈现 T1 → T2 → T3 上升：Claude/minimal 为 1 → 7 → 51，Codex/minimal 为 2 → 10 → 60，Claude/structured 为 0 → 5 → 45，Codex/structured 为 2 → 11 → 47。最大的增量都发生在 T2 → T3，说明 finding 数量主要随 T3 引入的前端和跨栈范围扩张，而不能简单解释为代码质量随任务推进等比例恶化。

在当前单次观测中，structured 条件的 T3 finding 少于同一 agent 的 minimal 条件（Claude：45 对 51；Codex：47 对 60）。

TODO： 增加的百分比是否有意义

## 3. Baseline 与 Agent × strategy 比较

先用一张汇总图比较 Baseline 与四个 Agent × strategy 条件（各条件累计 T1–T3），再用四张分阶段图分别展示 Baseline → T1 → T2 → T3。所有图均分开显示 Backend、Frontend 和 Cross-stack finding。

In [14]:
baseline_evaluation_path = REPORTS_DIR / 'baseline' / 'harness_evaluation.json'
if not baseline_evaluation_path.exists():
    raise FileNotFoundError(
        f'Baseline evaluation not found: {baseline_evaluation_path}. '
        'Run experiment/instruments/agent-runners/run_baseline_eval.py first.'
    )

baseline_evaluation = json.loads(baseline_evaluation_path.read_text(encoding='utf-8'))
baseline_subject_findings = {
    subject.get('subject_id'): len(
        ((subject.get('layers') or {}).get('constraints') or {}).get('findings') or []
    )
    for subject in baseline_evaluation.get('subjects') or []
}
baseline_cross_findings = len(
    ((((baseline_evaluation.get('cross_stack') or {}).get('layers') or {}).get('constraints') or {}).get('findings')) or []
)

condition_order = [
    ('claude', 'minimal'),
    ('claude', 'structured'),
    ('codex', 'minimal'),
    ('codex', 'structured'),
]
stage_categories = ['Baseline / Base', *tasks]
baseline_values = {
    'backend_findings': baseline_subject_findings.get('backend', 0),
    'frontend_findings': baseline_subject_findings.get('frontend', 0),
    'cross_findings': baseline_cross_findings,
}

condition_runs_by_name = {}
for agent, strategy in condition_order:
    condition_runs = [
        row for row in runs
        if row['agent'] == agent and row['strategy'] == strategy
    ]
    missing_tasks = [
        task for task in tasks
        if not any(row['task_id'] == task for row in condition_runs)
    ]
    if missing_tasks:
        raise ValueError(
            f'Missing evaluations for {agent}/{strategy}: {", ".join(missing_tasks)}'
        )

    condition_runs_by_name[(agent, strategy)] = condition_runs

summary_categories = [
    'Baseline / Base',
    *(f'{agent.title()} / {strategy}' for agent, strategy in condition_order),
]

def summary_values(field):
    return [
        baseline_values[field],
        *(
            sum(row[field] for row in condition_runs_by_name[(agent, strategy)])
            for agent, strategy in condition_order
        ),
    ]

display(HTML(ha.svg_grouped_bar(
    'Baseline 与 Agent × strategy 汇总（T1–T3 累计）',
    summary_categories,
    [
        ('Backend', summary_values('backend_findings')),
        ('Frontend', summary_values('frontend_findings')),
        ('Cross-stack', summary_values('cross_findings')),
    ],
    'Constraint findings (count)',
)))

for agent, strategy in condition_order:
    condition_runs = condition_runs_by_name[(agent, strategy)]

    def stage_values(field):
        return [
            baseline_values[field],
            *(
                mean(row[field] for row in condition_runs if row['task_id'] == task)
                for task in tasks
            ),
        ]

    display(HTML(ha.svg_grouped_bar(
        f'Baseline → {agent.title()}/{strategy}: T1–T3',
        stage_categories,
        [
            ('Backend', stage_values('backend_findings')),
            ('Frontend', stage_values('frontend_findings')),
            ('Cross-stack', stage_values('cross_findings')),
        ],
        'Constraint findings (count)',
    )))

### 讨论

汇总图中，Baseline 共发现 28 条约束问题（Backend 2、Frontend 24、Cross-stack 2）；T1–T3 累计后，Claude/minimal、Claude/structured、Codex/minimal 和 Codex/structured 分别为 59、50、72 和 60 条。四个实验条件的累计差异主要来自 Backend，Frontend 数量较接近。

需要注意，汇总图里的 Baseline 是一次初始快照，而每个 Agent × strategy 柱组是三个阶段的累计值，因此不能把柱高直接解释为相对 Baseline 的质量下降幅度。四张分阶段图更适合观察变化发生在哪个阶段；其中最大增量集中在 T3，也与 T3 新增前端和跨栈检查范围有关。

## 4. 规则热点矩阵

深色单元格表示该规则在对应 evaluation 中产生更多 finding

In [15]:
run_ids = [row['evaluation_id'] for row in runs]
run_labels = [f"{row['session_label']} {row['task_id']}" for row in runs]
rule_totals = Counter(row['rule_id'] for row in constraints)
top_rules = [rule for rule, _ in rule_totals.most_common(16)]
rule_lookup = Counter((row['rule_id'], row['evaluation_id']) for row in constraints)
rule_matrix = [
    [float(rule_lookup[(rule, evaluation_id)]) for evaluation_id in run_ids]
    for rule in top_rules
]
display(HTML(ha.svg_heatmap(
    '规则热点矩阵',
    top_rules,
    run_labels,
    rule_matrix,
    'finding count',
)))

### 讨论

热点高度集中：`FE-COM-C-002-jsx-max-depth`（79 条）和 `FE-STYLE-C-001-no-raw-jsx-style`（52 条）合计占全部 241 条 finding 的 54.4%，而且两者只在 T3 出现。

后端的 circular dependency（28 条）和 cross-module deep import（27 条）则从 T2 延续到 T3，更接近跨任务持续存在的架构边界问题。

—— T3 前端规模和规则适用范围带来的集中暴露；后两项更值得作为持续性重构候选。

## 5. 连续指标

不同 metric 的单位不可相加或直接比较。本图只在同一 metric 内归一化：0 是当前观测中相对较好，100 是相对较差；`higher_is_better` 已反向处理，灰色表示缺失或执行失败。

In [16]:
metric_labels, metric_matrix = ha.metric_badness(metrics, run_ids)
display(HTML(ha.svg_heatmap(
    '指标相对劣化矩阵（按指标内部归一化）',
    metric_labels,
    run_labels,
    metric_matrix,
    'relative badness (0–100)',
    maximum=100,
)))

### 讨论

灰色区域主要来自两类情况：部分前端 metric 只适用于 T3，以及 `BE-TEST-M-001-test-coverage` 的 7 次执行错误。

对有值的前端指标，四个 T3 session 的原始差距普遍较小，例如 component line average 为 78.95–82.63、JSX depth average 为 3.07–3.22；归一化后即使显示为 0 与 100，也不代表存在同等幅度的实质差异。

TODO: 查找 metrics 统计失败情况原因

## 6. 文件

In [17]:
file_counts = Counter(row['file'] for row in constraints if row.get('file'))
display(HTML(ha.svg_horizontal_bar(
    '跨 session 重复出现的问题文件',
    file_counts.most_common(15),
    'Findings across evaluations (count)',
)))

## 7. Harness 运行时间

运行时间用于观察任务复杂度和异常执行成本，不用于评价代码质量。

In [18]:
runtime_series = []
for session in sessions:
    condition = next(row for row in runs if row['session_id'] == session)
    label = f"{condition['agent']}/{condition['strategy']}"
    values = [
        (run_by_key[(session, task)]['duration_ms'] / 1000)
        if (session, task) in run_by_key and run_by_key[(session, task)]['duration_ms'] is not None
        else None
        for task in tasks
    ]
    runtime_series.append((label, values))

display(HTML(ha.svg_line_chart(
    'Harness 运行时长',
    tasks,
    runtime_series,
    'Duration (seconds)',
)))

### 讨论

所有条件的 T3 都是最耗时的阶段（4.517–5.922 秒），与 T3 扩展到前端和跨栈检查一致。三个任务累计耗时却非常接近：四个条件介于 11.145 与 11.733 秒之间，因此当前没有明显证据表明某个 agent 或 strategy 会系统性增加 Harness 成本。

运行时长反映被检查文件、启用规则和工具启动成本的组合，不是架构质量指标。由于每个点只有一次执行，也没有控制系统负载，亚秒级差异不应被解释为稳定的性能优势。

## 8. Metric 错误明细

Metric error 属于测量失败而非架构违规。修复 Harness 或 workspace 依赖后，应重新评估，再进行实验条件比较。

In [19]:
metric_errors = [row for row in metrics if row['status'] == 'error']
error_rows = ''.join(
    '<tr>'
    f"<td>{html.escape(row['evaluation_id'])}</td>"
    f"<td>{html.escape(row['subject_id'])}</td>"
    f"<td>{html.escape(row['metric_name'])}</td>"
    f"<td style='white-space:normal;max-width:700px'>{html.escape(row['findings'])}</td>"
    '</tr>'
    for row in metric_errors
) or '<tr><td colspan="4">No metric errors.</td></tr>'
display(HTML(
    '<table><thead><tr><th>Evaluation</th><th>Subject</th><th>Metric</th><th>Error evidence</th></tr></thead>'
    f'<tbody>{error_rows}</tbody></table>'
))

Evaluation,Subject,Metric,Error evidence
session_20260812_220814/T3,backend,BE-TEST-M-001-test-coverage,Runner crashed: Coverage command failed with code 1: ● Validation Error: Module ts-jest in the transform option was not found. <rootDir> is: /Users/luowei/project/ai-architecture-integrity-study/experiment/workspace/session_20260812_220814/backend/src Configuration Documentation: https://jestjs.io/docs/configuration
session_20260813_150337/T3,backend,BE-TEST-M-001-test-coverage,Runner crashed: Coverage command failed with code 1: ● Validation Error: Module ts-jest in the transform option was not found. <rootDir> is: /Users/luowei/project/ai-architecture-integrity-study/experiment/workspace/session_20260813_150337/backend/src Configuration Documentation: https://jestjs.io/docs/configuration
session_20260813_203443/T2,backend,BE-TEST-M-001-test-coverage,Runner crashed: Coverage command failed with code 1: ● Validation Error: Module ts-jest in the transform option was not found. <rootDir> is: /Users/luowei/project/ai-architecture-integrity-study/experiment/workspace/session_20260813_203443/backend/src Configuration Documentation: https://jestjs.io/docs/configuration
session_20260813_203443/T3,backend,BE-TEST-M-001-test-coverage,Runner crashed: Coverage command failed with code 1: ● Validation Error: Module ts-jest in the transform option was not found. <rootDir> is: /Users/luowei/project/ai-architecture-integrity-study/experiment/workspace/session_20260813_203443/backend/src Configuration Documentation: https://jestjs.io/docs/configuration
session_20260813_211609/T1,backend,BE-TEST-M-001-test-coverage,Runner crashed: Coverage command failed with code 1: ● Validation Error: Module ts-jest in the transform option was not found. <rootDir> is: /Users/luowei/project/ai-architecture-integrity-study/experiment/workspace/session_20260813_211609/backend/src Configuration Documentation: https://jestjs.io/docs/configuration
session_20260813_211609/T2,backend,BE-TEST-M-001-test-coverage,Runner crashed: Coverage command failed with code 1: ● Validation Error: Module ts-jest in the transform option was not found. <rootDir> is: /Users/luowei/project/ai-architecture-integrity-study/experiment/workspace/session_20260813_211609/backend/src Configuration Documentation: https://jestjs.io/docs/configuration
session_20260813_211609/T3,backend,BE-TEST-M-001-test-coverage,Runner crashed: Coverage command failed with code 1: ● Validation Error: Module ts-jest in the transform option was not found. <rootDir> is: /Users/luowei/project/ai-architecture-integrity-study/experiment/workspace/session_20260813_211609/backend/src Configuration Documentation: https://jestjs.io/docs/configuration


### 讨论

7 个 metric error 全部来自同一个测量项 `BE-TEST-M-001-test-coverage`，错误证据也一致：Jest 无法解析 `ts-jest`。这是一项系统性的依赖/配置故障，而不是 7 个独立的架构问题，也不能把缺失的 coverage 当作低 coverage。

错误分布并不均匀：Codex/structured 的 T1–T3 均受影响，Claude/structured 的 T2–T3 受影响，两个 minimal 条件仅 T3 受影响。若直接比较综合 metric，条件差异会被不对称缺失所混淆；应先修复 `ts-jest` 解析路径并重跑这 7 次测量。

## 9. 导出

 Notebook 使用的数据导出到 `reports/analysis/notebook_output/`

In [20]:
output_dir = ANALYSIS_DIR / 'notebook_output'
output_dir.mkdir(parents=True, exist_ok=True)
ha.write_csv(output_dir / 'run_summary.csv', runs)
ha.write_csv(output_dir / 'constraint_findings.csv', constraints)
ha.write_csv(output_dir / 'metric_values.csv', metrics)
ha.write_csv(output_dir / 'file_hotspots.csv', ha.build_file_hotspots(constraints))
print(f'Exported notebook data to: {output_dir}')

Exported notebook data to: /Users/luowei/project/ai-architecture-integrity-study/reports/analysis/notebook_output
